In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
load_dotenv(override=True)
url = os.getenv("MYSQL_URL")

In [2]:
import db_utils
import importlib

importlib.reload(db_utils)

<module 'db_utils' from 'c:\\Users\\Dell\\Data_Engineering_Workspace\\db_utils.py'>

In [3]:
df = pd.read_csv("Superstore_utf8.csv")
df.shape

(9994, 21)

In [5]:
db_utils.run_mysql_query("""
SHOW CREATE TABLE superstore.staging_sales;
""")

'Execution successful. Rows affected: 1'

In [6]:
db_utils.run_mysql_query("""
SELECT 
    TABLE_NAME,
    TABLE_ROWS
FROM information_schema.tables
WHERE TABLE_SCHEMA = 'superstore'
AND TABLE_NAME = 'staging_sales';
""")

,TABLE_NAME,TABLE_ROWS
0,staging_sales,0


In [7]:
db_utils.run_mysql_query("""
SELECT 
    COLUMN_NAME,
    COLUMN_TYPE,
    IS_NULLABLE,
    COLUMN_KEY
FROM information_schema.columns
WHERE TABLE_SCHEMA = 'superstore'
AND TABLE_NAME = 'staging_sales'
ORDER BY ORDINAL_POSITION;
""")

,COLUMN_NAME,COLUMN_TYPE,IS_NULLABLE,COLUMN_KEY
0,row_id,int,NO,PRI
1,order_id,varchar(20),YES,
2,order_date,date,YES,
3,ship_date,date,YES,
4,ship_mode,varchar(20),YES,
5,customer_id,varchar(20),YES,
6,customer_name,varchar(100),YES,
7,segment,varchar(50),YES,
8,country,varchar(100),YES,
9,city,varchar(100),YES,


In [8]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [9]:
staging_df = df.rename(columns={
    "Row ID": "row_id",
    "Order ID": "order_id",
    "Order Date": "order_date",
    "Ship Date": "ship_date",
    "Ship Mode": "ship_mode",
    "Customer ID": "customer_id",
    "Customer Name": "customer_name",
    "Segment": "segment",
    "Country": "country",
    "City": "city",
    "State": "state",
    "Postal Code": "postal_code",
    "Region": "region",
    "Product ID": "product_id",
    "Category": "category",
    "Sub-Category": "sub_category",
    "Product Name": "product_name",
    "Sales": "sales",
    "Quantity": "quantity",
    "Discount": "discount",
    "Profit": "profit"
})

In [10]:
staging_df["order_date"] = pd.to_datetime(staging_df["order_date"])
staging_df["ship_date"] = pd.to_datetime(staging_df["ship_date"])

In [11]:
staging_df.shape

(9994, 21)

In [12]:
staging_df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [13]:
import os
from sqlalchemy import create_engine
load_dotenv(override=True)

mysql_url = os.getenv("MYSQL_URL")

engine = create_engine(
    mysql_url,
    pool_pre_ping=True
)
with engine.connect() as conn:
    print("MySQL connection successful!")

MySQL connection successful!


In [15]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE()"))
    print(result.fetchone())

('defaultdb',)


In [16]:
from sqlalchemy import create_engine
from sqlalchemy.engine import make_url

url = make_url(mysql_url)

superstore_url = url.set(database="superstore")

engine_superstore = create_engine(
    superstore_url,
    pool_pre_ping=True
)

In [17]:
with engine_superstore.connect() as conn:
    result = conn.execute(text("SELECT DATABASE()"))
    print(result.fetchone())

('superstore',)


In [18]:
staging_df.to_sql(
    "staging_sales",
    con=engine_superstore,
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

9994

In [19]:
db_utils.run_mysql_query("""
SELECT COUNT(*) AS row_count
FROM superstore.staging_sales;
""")

,row_count
0,9994


In [20]:
db_utils.run_mysql_query("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT row_id) AS unique_row_ids
FROM superstore.staging_sales;
""")

,total_rows,unique_row_ids
0,9994,9994


In [21]:
db_utils.run_mysql_query("""
SELECT
    COUNT(*) AS total_rows,
    SUM(row_id IS NULL) AS null_row_id,
    SUM(order_id IS NULL) AS null_order_id,
    SUM(customer_id IS NULL) AS null_customer_id,
    SUM(product_id IS NULL) AS null_product_id,
    SUM(sales IS NULL) AS null_sales,
    SUM(quantity IS NULL) AS null_quantity,
    SUM(profit IS NULL) AS null_profit
FROM superstore.staging_sales;
""")

,total_rows,null_row_id,null_order_id,null_customer_id,null_product_id,null_sales,null_quantity,null_profit
0,9994,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
!git pull origin main

remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 19 (delta 7), reused 19 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (19/19), 5.67 MiB | 15.00 KiB/s, done.
From https://github.com/leodera/Data_Engineering_Workspace
 * branch            main       -> FETCH_HEAD
   431eded..3e4a73e  main       -> origin/main
Updating 431eded..3e4a73e
error: Your local changes to the following files would be overwritten by merge:
	Untitled1.ipynb
	nano_02_rfm_quantiles.ipynb
Please commit your changes or stash them before you merge.
error: The following untracked working tree files would be overwritten by merge:
	.ipynb_checkpoints/data_cleaning-checkpoint.ipynb
	.ipynb_checkpoints/db_utils-checkpoint.py
	.ipynb_checkpoints/nano_02_rfm_quantiles-checkpoint.ipynb
	data_cleaning.ipynb
	db_utils.py
Please move or remove them before you merge.
Aborting


In [20]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(override=True)

mysql_engine = create_engine(
    os.getenv("MYSQL_URL"),
    pool_pre_ping=True
)

postgres_engine = create_engine(
    os.getenv("POSTGRES_URL"),
    pool_pre_ping=True
)

In [24]:
with mysql_engine.connect() as conn:
    print("MySQL:", conn.execute(text("SELECT DATABASE()")).scalar())

with postgres_engine.connect() as conn:
    print("PostgreSQL:", conn.execute(text("SELECT current_database()")).scalar())

MySQL: defaultdb
PostgreSQL: defaultdb


In [25]:
from sqlalchemy.engine import make_url

url = make_url(os.getenv("MYSQL_URL"))

superstore_mysql_url = url.set(database="superstore")

mysql_engine = create_engine(
    superstore_mysql_url,
    pool_pre_ping=True
)

In [26]:
with mysql_engine.connect() as conn:
    print(
        "MySQL database:",
        conn.execute(text("SELECT DATABASE()")).scalar()
    )

MySQL database: superstore


In [27]:
db_utils.run_pg_query("""
CREATE SCHEMA IF NOT EXISTS superstore;
""")

'Execution successful. Rows affected: -1'

In [28]:
db_utils.run_pg_query("""
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name = 'superstore';
""")

,schema_name
0,superstore


In [29]:
db_utils.run_pg_query("""
CREATE TABLE IF NOT EXISTS superstore.customers (
    customer_id VARCHAR(20) PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    segment VARCHAR(50),
    country VARCHAR(100)
);
""")

db_utils.run_pg_query("""
CREATE TABLE IF NOT EXISTS superstore.products (
    product_key INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    product_id VARCHAR(20) NOT NULL,
    product_name VARCHAR(255) NOT NULL,
    category VARCHAR(100) NOT NULL,
    sub_category VARCHAR(100) NOT NULL
);
""")

db_utils.run_pg_query("""
CREATE TABLE IF NOT EXISTS superstore.orders (
    order_id VARCHAR(20) PRIMARY KEY,
    customer_id VARCHAR(20) NOT NULL,
    order_date DATE NOT NULL,
    ship_date DATE NOT NULL,
    ship_mode VARCHAR(20) NOT NULL,
    city VARCHAR(100) NOT NULL,
    state VARCHAR(100) NOT NULL,
    postal_code VARCHAR(10) NOT NULL,
    region VARCHAR(20) NOT NULL,

    CONSTRAINT fk_orders_customer
        FOREIGN KEY (customer_id)
        REFERENCES superstore.customers(customer_id)
);
""")

db_utils.run_pg_query("""
CREATE TABLE IF NOT EXISTS superstore.order_items (
    row_id INTEGER PRIMARY KEY,
    order_id VARCHAR(20) NOT NULL,
    product_key INTEGER NOT NULL,
    sales NUMERIC(15,4) NOT NULL,
    quantity INTEGER NOT NULL,
    discount NUMERIC(5,4) NOT NULL,
    profit NUMERIC(15,4) NOT NULL,

    CONSTRAINT fk_order_items_order
        FOREIGN KEY (order_id)
        REFERENCES superstore.orders(order_id),

    CONSTRAINT fk_order_items_product
        FOREIGN KEY (product_key)
        REFERENCES superstore.products(product_key)
);
""")

'Execution successful. Rows affected: -1'

In [32]:
db_utils.run_pg_query("""
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'superstore'
ORDER BY table_name;
""")

,table_schema,table_name
0,superstore,customers
1,superstore,order_items
2,superstore,orders
3,superstore,products
4,superstore,sample_superstore
5,superstore,v_super


In [36]:
from sqlalchemy.engine import make_url

mysql_url = make_url(os.getenv("MYSQL_URL"))
mysql_superstore_url = mysql_url.set(database="superstore")

mysql_engine = create_engine(
    mysql_superstore_url,
    pool_pre_ping=True
)

with mysql_engine.connect() as conn:
    print(conn.execute(text("SELECT DATABASE()")).scalar())

customers_df = pd.read_sql(
    "SELECT * FROM Customers",
    mysql_engine
)

products_df = pd.read_sql(
    "SELECT * FROM Products",
    mysql_engine
)

orders_df = pd.read_sql(
    "SELECT * FROM Orders",
    mysql_engine
)

order_items_df = pd.read_sql(
    "SELECT * FROM Order_Items",
    mysql_engine
)

customers_df.to_sql(
    "customers",
    con=postgres_engine,
    schema="superstore",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

superstore


793

In [39]:
products_df[["product_key", "product_id", "product_name"]].head() 

print("Customers:", customers_df.shape)
print("Products:", products_df.shape)
print("Orders:", orders_df.shape)
print("Order Items:", order_items_df.shape)

Customers: (793, 4)
Products: (1894, 5)
Orders: (5009, 9)
Order Items: (9994, 7)


In [40]:
products_df.to_sql(
    "products",
    con=postgres_engine,
    schema="superstore",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

ProgrammingError: (psycopg2.errors.GeneratedAlways) cannot insert a non-DEFAULT value into column "product_key"
DETAIL:  Column "product_key" is an identity column defined as GENERATED ALWAYS.
HINT:  Use OVERRIDING SYSTEM VALUE to override.

[SQL: INSERT INTO superstore.products (product_key, product_id, product_name, category, sub_category) VALUES (%(product_key_m0)s, %(product_id_m0)s, %(product_name_m0)s, %(category_m0)s, %(sub_category_m0)s), (%(product_key_m1)s, %(product_id_m1)s, %(product_name_m1)s, %(category_m1)s, %(sub_category_m1)s), (%(product_key_m2)s, %(product_id_m2)s, %(product_name_m2)s, %(category_m2)s, %(sub_category_m2)s), (%(product_key_m3)s, %(product_id_m3)s, %(product_name_m3)s, %(category_m3)s, %(sub_category_m3)s), (%(product_key_m4)s, %(product_id_m4)s, %(product_name_m4)s, %(category_m4)s, %(sub_category_m4)s), (%(product_key_m5)s, %(product_id_m5)s, %(product_name_m5)s, %(category_m5)s, %(sub_category_m5)s), (%(product_key_m6)s, %(product_id_m6)s, %(product

In [41]:
db_utils.run_pg_query("""
ALTER TABLE superstore.products
ALTER COLUMN product_key
SET GENERATED BY DEFAULT;
""")

'Execution successful. Rows affected: -1'

In [42]:
db_utils.run_pg_query("""
SELECT
    column_name,
    is_identity,
    identity_generation
FROM information_schema.columns
WHERE table_schema = 'superstore'
AND table_name = 'products'
AND column_name = 'product_key';
""")

,column_name,is_identity,identity_generation
0,product_key,YES,BY DEFAULT


In [43]:
products_df.to_sql(
    "products",
    con=postgres_engine,
    schema="superstore",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

1894

In [44]:
db_utils.run_pg_query("""
SELECT
    COUNT(*) AS product_count,
    MIN(product_key) AS min_product_key,
    MAX(product_key) AS max_product_key
FROM superstore.products;
""")

,product_count,min_product_key,max_product_key
0,1894,1,1894


In [45]:
db_utils.run_pg_query("""
SELECT setval(
    pg_get_serial_sequence('superstore.products', 'product_key'),
    (SELECT MAX(product_key) FROM superstore.products)
);
""")

,setval
0,1894


In [46]:
db_utils.run_pg_query("""
SELECT
    COUNT(*) AS product_count,
    MIN(product_key) AS min_key,
    MAX(product_key) AS max_key
FROM superstore.products;
""")

,product_count,min_key,max_key
0,1894,1,1894


In [47]:
orders_df.to_sql(
    "orders",
    con=postgres_engine,
    schema="superstore",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

5009

In [48]:
db_utils.run_pg_query("""
SELECT COUNT(*) AS product_count
FROM superstore.products;
""")

,product_count
0,1894


In [49]:
order_items_df.to_sql(
    "order_items",
    con=postgres_engine,
    schema="superstore",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

9994

In [50]:
db_utils.run_pg_query("""
SELECT COUNT(*) AS order_item_count
FROM superstore.order_items;
""")

,order_item_count
0,9994


In [51]:
mysql_totals = pd.read_sql("""
SELECT
    SUM(sales) AS sales,
    SUM(quantity) AS quantity,
    SUM(discount) AS discount,
    SUM(profit) AS profit
FROM Order_Items;
""", mysql_engine)

postgres_totals = pd.read_sql("""
SELECT
    SUM(sales) AS sales,
    SUM(quantity) AS quantity,
    SUM(discount) AS discount,
    SUM(profit) AS profit
FROM superstore.order_items;
""", postgres_engine)

print("MYSQL")
display(mysql_totals)

print("POSTGRESQL")
display(postgres_totals)

MYSQL


,sales,quantity,discount,profit
0,2297201.07,37873.0,1561.09,286397.0217


POSTGRESQL


,sales,quantity,discount,profit
0,2297201.07,37873,1561.09,286397.0217


In [52]:
comparison = pd.DataFrame({
    "MySQL": mysql_totals.iloc[0],
    "PostgreSQL": postgres_totals.iloc[0]
})

comparison["Difference"] = (
    comparison["PostgreSQL"] - comparison["MySQL"]
)

display(comparison)

,MySQL,PostgreSQL,Difference
sales,2.297201e+06,2.297201e+06,0.0
quantity,3.787300e+04,3.787300e+04,0.0
discount,1.561090e+03,1.561090e+03,0.0
profit,2.863970e+05,2.863970e+05,0.0


## BUSINESS CASE 1: ARE WE ACTUALLY MAKING MONEY?

### Management Request

> "Give me a high-level performance overview of the business. I want to know how much revenue we've generated, how much profit we've made, and our overall profit margin."

In [6]:
query = """
SELECT
    SUM(sales) AS "Total Revenue",
    SUM(profit) AS "Total Profit",
    ROUND(
        (SUM(profit) / NULLIF(SUM(sales), 0)) * 100,
        2
    ) AS "Profit Margin %"
FROM superstore.order_items;
"""

In [7]:
db_utils.run_pg_query(query)

,Total Revenue,Total Profit,Profit Margin %
0,2297201.07,286397.0217,12.47


### Performance Summary

* **Total Revenue:** $2,297,201.07
* **Total Profit:** $286,397.02
* **Profit Margin:** 12.47%

> **Key Takeaway:** The business remains profitable with a **12.47%** margin, yielding **$286,397.02** in net profit from **$2.30M** in revenue.

## BUSINESS CASE 2: PRODUCT PERFORMANCE ANALYSIS

### Management Request
> "Which product categories are driving our profitability, and which ones should we be concerned about?"

---

### Executive Performance Breakdown

| Category | Total Revenue | Total Profit | Profit Margin % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **Technology** | $836,154.10 | $145,454.95 | 17.40% | 🟢 Top Driver |
| **Office Supplies** | $719,046.99 | $122,490.80 | 17.04% | 🟢 Strong Performer |
| **Furniture** | $741,999.98 | $18,451.27 | 2.49% | 🔴 High Concern |

---

### Key Takeaways & Strategic Recommendations

* **Drivers of Profitability:** **Technology** and **Office Supplies** drive **93.5%** of the business's total profits, maintaining strong profit margins near **17%**.
* **Primary Concern:** **Furniture** is severely underperforming. Despite bringing in **$741.99K** in revenue, it yields only **$18.45K** in profit (**2.49%** margin). 
* **Next Steps:** Investigate discounting patterns, shipping/logistics costs, and product-level margins within the Furniture sub-categories to identify where margins are being eroded.

In [8]:
query2 = """
SELECT
    p.category AS Category,
    SUM(o.sales) AS "Total Revenue",
    SUM(o.profit) AS "Total Profit",
    ROUND(
        (SUM(o.profit) / NULLIF(SUM(o.sales), 0)) * 100,
        2
    ) AS "Profit Margin %"
FROM superstore.order_items o
JOIN superstore.products p
    ON p.product_key = o.product_key
GROUP BY p.category;
"""

db_utils.run_pg_query(query2)

,category,Total Revenue,Total Profit,Profit Margin %
0,Furniture,741999.98,18451.2728,2.49
1,Office Supplies,719046.99,122490.8008,17.04
2,Technology,836154.10,145454.9481,17.40


## BUSINESS CASE 3: INVESTIGATING FURNITURE SUB-CATEGORIES

### Management Request
> "Furniture has a much lower profit margin than the other categories. Find out which sub-categories are responsible for this poor performance."

---

### Furniture Sub-Category Performance

| Sub-Category | Total Revenue | Total Profit | Profit Margin % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **Furnishings** | $91,705.12 | $13,059.14 | 14.24% | 🟢 Healthy |
| **Chairs** | $328,449.13 | $26,590.17 | 8.10% | 🟡 Moderate |
| **Bookcases** | $114,880.05 | -$3,472.56 | -3.02% | 🔴 Net Loss |
| **Tables** | $206,965.68 | -$17,725.48 | -8.56% | 🔴 Major Drag |

---

### Key Findings & Root Cause

* **Root Cause Identified:** The low Furniture margin isn't widespread—it is heavily driven by **Tables** (-8.56% margin) and **Bookcases** (-3.02% margin), which together lost **-$21,198.04**.
* **Profit Cannibalization:** Strong profits from **Chairs** ($26.59K) and high-margin **Furnishings** (14.24%) are being eroded to compensate for net losses in Tables and Bookcases.

---

### Strategic Recommendations
1. **Discount Controls:** Re-evaluate promotional strategy and set strict discount caps on Tables and Bookcases.
2. **Cost Structure & Freight:** Audit shipping and freight costs for heavy goods (Tables/Bookcases) to ensure delivery costs aren't exceeding margin thresholds.
3. **Product Line Audit:** Re-assess vendor costs or prune unprofitable SKU items within the Tables sub-category.

In [9]:
query3 = """
SELECT
    p.sub_category AS "Sub-Category",
    SUM(i.sales) AS "Total Revenue",
    SUM(i.profit) AS "Total Profit",
    ROUND(
        (SUM(i.profit) / NULLIF(SUM(i.sales), 0)) * 100,
        2
    ) AS "Profit Margin %"
FROM superstore.order_items i
JOIN superstore.products p
    ON i.product_key = p.product_key
WHERE p.category = 'Furniture'
GROUP BY p.sub_category
ORDER BY SUM(i.profit) ASC;
"""
db_utils.run_pg_query(query3)

,Sub-Category,Total Revenue,Total Profit,Profit Margin %
0,Tables,206965.68,-17725.4811,-8.56
1,Bookcases,114880.05,-3472.5560,-3.02
2,Furnishings,91705.12,13059.1436,14.24
3,Chairs,328449.13,26590.1663,8.10


# BUSINESS CASE 4: CUSTOMER PROFITABILITY ANALYSIS

## Management Request

> "We know some customers are unprofitable. Which customers are generating losses, and which product categories are responsible for those losses?"

### Business Question

We want to determine:

1. Which customers are unprofitable overall?
2. How much revenue are these customers generating?
3. How much profit/loss are they generating?
4. Which categories are responsible for their losses?
5. Which customers represent the greatest financial risk?

### Analytical Approach

We will:

1. Calculate total profit for each customer.
2. Identify customers whose total profit is below zero.
3. Break those customers down by product category.
4. Identify categories where each unprofitable customer is losing money.
5. Quantify the losses from those unprofitable categories.
6. Prioritize customers based on the magnitude of their losses.

### Important Business Rule

A customer is considered **unprofitable** when:

`Total Customer Profit < 0`

A category is considered **unprofitable for a customer** when:

`Customer + Category Profit < 0`

QUERY 1 - UNPROFITABLE CUSTOMERS

In [15]:
query = """
WITH unprofitable_customers AS (
    SELECT
        o.customer_id
    FROM superstore.orders o
    JOIN superstore.order_items i
        ON o.order_id = i.order_id
    GROUP BY
        o.customer_id
    HAVING SUM(i.profit) < 0
)
SELECT *
FROM unprofitable_customers;
"""
db_utils.run_pg_query(query)


,customer_id
0,TS-21610
1,EM-13810
2,JR-15670
3,JH-16180
4,SB-20290
...,...
150,EH-14005
151,ZC-21910
152,SP-20620
153,MD-17350


Query 2 — Unprofitable customers by category

In [17]:
query = """
WITH unprofitable_customers AS (
    SELECT
        o.customer_id
    FROM superstore.orders o
    JOIN superstore.order_items i
        ON o.order_id = i.order_id
    GROUP BY
        o.customer_id
    HAVING SUM(i.profit) < 0
)

SELECT
    c.customer_name,
    p.category,
    SUM(i.sales) AS "Total Revenue",
    SUM(i.profit) AS "Total Profit",
    ROUND(
        (SUM(i.profit) / NULLIF(SUM(i.sales), 0)) * 100,
        2
    ) AS "Profit Margin %"
FROM superstore.customers c
JOIN superstore.orders o
    ON c.customer_id = o.customer_id
JOIN superstore.order_items i
    ON i.order_id = o.order_id
JOIN superstore.products p
    ON i.product_key = p.product_key
WHERE c.customer_id IN (
    SELECT customer_id
    FROM unprofitable_customers
)
GROUP BY
    c.customer_name,
    p.category
ORDER BY
    SUM(i.profit) ASC;
    """
db_utils.run_pg_query(query)

,customer_name,category,Total Revenue,Total Profit,Profit Margin %
0,Cindy Stewart,Technology,4955.97,-6397.9028,-129.09
1,Grant Thornton,Technology,7999.98,-3839.9904,-48.00
2,Luke Foster,Office Supplies,2945.39,-3611.9877,-122.63
3,Sharelle Roach,Technology,2549.99,-3399.9800,-133.33
4,Henry Goldwyn,Office Supplies,2993.09,-2854.2811,-95.36
...,...,...,...,...,...
420,Joseph Holt,Technology,1031.89,413.5968,40.08
421,Zuschuss Carroll,Furniture,3899.69,528.6897,13.56
422,Victoria Wilson,Office Supplies,1982.58,578.9763,29.20
423,Laurel Beltran,Office Supplies,3110.14,748.8433,24.08


Query 3 — Final customer-level loss analysis

In [12]:
query = """
WITH unprofitable_customers AS (
    SELECT
        o.customer_id
    FROM superstore.orders o
    JOIN superstore.order_items i
        ON o.order_id = i.order_id
    GROUP BY
        o.customer_id
    HAVING SUM(i.profit) < 0
),

customer_category_profit AS (
    SELECT
        o.customer_id,
        p.category,
        SUM(i.sales) AS category_revenue,
        SUM(i.profit) AS category_profit
    FROM superstore.orders o
    JOIN superstore.order_items i
        ON o.order_id = i.order_id
    JOIN superstore.products p
        ON i.product_key = p.product_key
    WHERE o.customer_id IN (
        SELECT customer_id
        FROM unprofitable_customers
    )
    GROUP BY
        o.customer_id,
        p.category
)

SELECT
    c.customer_name,
    SUM(ccp.category_revenue) AS "Total Revenue",
    SUM(ccp.category_profit) AS "Total Profit",

    SUM(
        CASE
            WHEN ccp.category_profit < 0
            THEN ccp.category_profit
            ELSE 0
        END
    ) AS "Loss From Unprofitable Categories",

    COUNT(
        CASE
            WHEN ccp.category_profit < 0
            THEN 1
        END
    ) AS "Unprofitable Categories"

FROM customer_category_profit ccp
JOIN superstore.customers c
    ON c.customer_id = ccp.customer_id

GROUP BY
    c.customer_id,
    c.customer_name

ORDER BY
    "Total Profit" ASC;
    """
db_utils.run_pg_query(query)

,customer_name,Total Revenue,Total Profit,Loss From Unprofitable Categories,Unprofitable Categories
0,Cindy Stewart,5690.07,-6626.3895,-6626.3895,2
1,Grant Thornton,9351.20,-4108.6589,-4187.1078,2
2,Luke Foster,3930.52,-3583.9770,-3638.9337,2
3,Sharelle Roach,3233.48,-3333.9144,-3399.9800,1
4,Henry Goldwyn,3247.65,-2797.9635,-2860.4691,2
...,...,...,...,...,...
150,Thais Sissman,4.84,-3.3156,-3.3156,1
151,Adrian Hane,1735.53,-2.3146,-296.5673,2
152,Mitch Gastineau,16.74,-1.2453,-3.0933,1
153,Paul Lucas,239.49,-0.7527,-13.6461,1


Query 4 — Rank the worst customers

In [13]:
query = """
WITH unprofitable_customers AS (
    SELECT
        o.customer_id
    FROM superstore.orders o
    JOIN superstore.order_items i
        ON o.order_id = i.order_id
    GROUP BY
        o.customer_id
    HAVING SUM(i.profit) < 0
),

customer_profit AS (
    SELECT
        o.customer_id,
        SUM(i.sales) AS total_revenue,
        SUM(i.profit) AS total_profit
    FROM superstore.orders o
    JOIN superstore.order_items i
        ON o.order_id = i.order_id
    WHERE o.customer_id IN (
        SELECT customer_id
        FROM unprofitable_customers
    )
    GROUP BY
        o.customer_id
)

SELECT
    c.customer_name,
    cp.total_revenue AS "Total Revenue",
    cp.total_profit AS "Total Profit",
    RANK() OVER (
        ORDER BY cp.total_profit ASC
    ) AS loss_rank
FROM customer_profit cp
JOIN superstore.customers c
    ON c.customer_id = cp.customer_id
ORDER BY
    loss_rank;
    """
db_utils.run_pg_query(query)

,customer_name,Total Revenue,Total Profit,loss_rank
0,Cindy Stewart,5690.07,-6626.3895,1
1,Grant Thornton,9351.20,-4108.6589,2
2,Luke Foster,3930.52,-3583.9770,3
3,Sharelle Roach,3233.48,-3333.9144,4
4,Henry Goldwyn,3247.65,-2797.9635,5
...,...,...,...,...
150,Thais Sissman,4.84,-3.3156,151
151,Adrian Hane,1735.53,-2.3146,152
152,Mitch Gastineau,16.74,-1.2453,153
153,Paul Lucas,239.49,-0.7527,154


Query 5 — Find the categories causing the biggest customer losses

In [14]:
query = """
WITH customer_category_profit AS (
    SELECT
        o.customer_id,
        p.category,
        SUM(i.sales) AS total_revenue,
        SUM(i.profit) AS total_profit
    FROM superstore.orders o
    JOIN superstore.order_items i
        ON o.order_id = i.order_id
    JOIN superstore.products p
        ON i.product_key = p.product_key
    GROUP BY
        o.customer_id,
        p.category
)

SELECT
    c.customer_name,
    ccp.category,
    ccp.total_revenue AS "Total Revenue",
    ccp.total_profit AS "Total Profit",
    ROUND(
        ccp.total_profit / NULLIF(ccp.total_revenue, 0) * 100,
        2
    ) AS "Profit Margin %"
FROM customer_category_profit ccp
JOIN superstore.customers c
    ON c.customer_id = ccp.customer_id
WHERE ccp.total_profit < 0
ORDER BY
    ccp.total_profit ASC;
    """
db_utils.run_pg_query(query)

,customer_name,category,Total Revenue,Total Profit,Profit Margin %
0,Cindy Stewart,Technology,4955.97,-6397.9028,-129.09
1,Grant Thornton,Technology,7999.98,-3839.9904,-48.00
2,Luke Foster,Office Supplies,2945.39,-3611.9877,-122.63
3,Sharelle Roach,Technology,2549.99,-3399.9800,-133.33
4,Henry Goldwyn,Office Supplies,2993.09,-2854.2811,-95.36
...,...,...,...,...,...
472,Larry Blacks,Furniture,1.99,-1.4413,-72.43
473,Maurice Satty,Technology,122.95,-1.1528,-0.94
474,Stephanie Phelps,Technology,1013.01,-1.1444,-0.11
475,Guy Thornton,Furniture,35.54,-0.8886,-2.50


Business Case 5 — Monthly Performance Trend

In [18]:
query = """
SELECT
    DATE_TRUNC('month', o.order_date) AS month,
    SUM(i.sales) AS "Total Revenue",
    SUM(i.profit) AS "Total Profit",
    ROUND(
        (SUM(i.profit) / NULLIF(SUM(i.sales), 0)) * 100,
        2
    ) AS "Profit Margin %"
FROM superstore.orders o
JOIN superstore.order_items i
    ON o.order_id = i.order_id
GROUP BY
    DATE_TRUNC('month', o.order_date)
ORDER BY
    month;
    """
db_utils.run_pg_query(query)

,month,Total Revenue,Total Profit,Profit Margin %
0,2014-01-01 00:00:00+00:00,14236.90,2450.1907,17.21
1,2014-02-01 00:00:00+00:00,4519.92,862.3084,19.08
2,2014-03-01 00:00:00+00:00,55691.04,498.7299,0.90
3,2014-04-01 00:00:00+00:00,28295.35,3488.8352,12.33
4,2014-05-01 00:00:00+00:00,23648.28,2738.7096,11.58
5,2014-06-01 00:00:00+00:00,34595.14,4976.5244,14.39
6,2014-07-01 00:00:00+00:00,33946.37,-841.4826,-2.48
7,2014-08-01 00:00:00+00:00,27909.47,5318.1050,19.05
8,2014-09-01 00:00:00+00:00,81777.34,8328.0994,10.18
9,2014-10-01 00:00:00+00:00,31453.37,3448.2573,10.96


# EXECUTIVE DASHBOARD DATASETS

In [19]:
# ============================================
# EXECUTIVE DASHBOARD
# ============================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# KPI DATASET

In [39]:
# ============================================
# KPI SUMMARY
# ============================================

kpi_query = """
SELECT
    SUM(sales) AS "Total Revenue",
    SUM(profit) AS "Total Profit",
    ROUND(
        (SUM(profit) / NULLIF(SUM(sales), 0)) * 100,
        2
    ) AS "Profit Margin %",
    COUNT(DISTINCT order_id) AS "Total Orders"
FROM superstore.order_items;
"""

kpi_df = db_utils.run_pg_query(kpi_query)

kpi_df

,Total Revenue,Total Profit,Profit Margin %,Total Orders
0,2297201.07,286397.0217,12.47,5009


## Orders By Category

In [40]:
orders_category_query = """
SELECT
    p.category AS "Category",
    COUNT(DISTINCT i.order_id) AS "Total Orders"
FROM superstore.order_items i
JOIN superstore.products p
    ON i.product_key = p.product_key
GROUP BY p.category
ORDER BY "Total Orders" DESC;
"""

orders_category_df = db_utils.run_pg_query(orders_category_query)

orders_category_df

,Category,Total Orders
0,Office Supplies,3742
1,Furniture,1764
2,Technology,1544


## Orders by region

In [41]:
orders_region_query = """
SELECT
    o.region AS "Region",
    COUNT(DISTINCT o.order_id) AS "Total Orders"
FROM superstore.orders o
GROUP BY o.region
ORDER BY "Total Orders" DESC;
"""

orders_region_df = db_utils.run_pg_query(orders_region_query)

orders_region_df

,Region,Total Orders
0,West,1611
1,East,1401
2,Central,1175
3,South,822


## Orders by state

In [42]:
orders_state_query = """
SELECT
    o.state AS "State",
    COUNT(DISTINCT o.order_id) AS "Total Orders"
FROM superstore.orders o
GROUP BY o.state
ORDER BY "Total Orders" DESC;
"""

orders_state_df = db_utils.run_pg_query(orders_state_query)

orders_state_df

,State,Total Orders
0,California,1021
1,New York,562
2,Texas,487
3,Pennsylvania,288
4,Illinois,276
5,Washington,256
6,Ohio,236
7,Florida,200
8,North Carolina,136
9,Michigan,117


## Orders By Customer

In [43]:
orders_customer_query = """
SELECT
    c.customer_name AS "Customer Name",
    COUNT(DISTINCT o.order_id) AS "Total Orders"
FROM superstore.orders o
JOIN superstore.customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY "Total Orders" DESC
LIMIT 10;
"""

orders_customer_df = db_utils.run_pg_query(orders_customer_query)

orders_customer_df

,Customer Name,Total Orders
0,Emily Phan,17
1,Chloris Kastensmidt,13
2,Joel Eaton,13
3,Noel Staavos,13
4,Sally Hughsby,13
5,Zuschuss Carroll,13
6,Patrick Gardner,13
7,Erin Ashbrook,13
8,Chris Selesnick,12
9,Bart Pistole,12


## CATEGORY PERFORMANCE

In [23]:
category_df = db_utils.run_pg_query("""
SELECT
    p.category,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.order_items i
JOIN superstore.products p
    ON i.product_key = p.product_key
GROUP BY
    p.category
ORDER BY
    profit DESC;
""")

category_df

,category,revenue,profit,profit_margin
0,Technology,836154.10,145454.9481,17.40
1,Office Supplies,719046.99,122490.8008,17.04
2,Furniture,741999.98,18451.2728,2.49


## Furniture sub-category performance

In [24]:
furniture_df = db_utils.run_pg_query("""
SELECT
    p.sub_category,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.order_items i
JOIN superstore.products p
    ON i.product_key = p.product_key
WHERE
    p.category = 'Furniture'
GROUP BY
    p.sub_category
ORDER BY
    profit ASC;
""")

furniture_df

,sub_category,revenue,profit,profit_margin
0,Tables,206965.68,-17725.4811,-8.56
1,Bookcases,114880.05,-3472.5560,-3.02
2,Furnishings,91705.12,13059.1436,14.24
3,Chairs,328449.13,26590.1663,8.10


## Monthly Performance

In [25]:
monthly_df = db_utils.run_pg_query("""
SELECT
    DATE_TRUNC('month', o.order_date) AS month,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.orders o
JOIN superstore.order_items i
    ON o.order_id = i.order_id
GROUP BY
    DATE_TRUNC('month', o.order_date)
ORDER BY
    month;
""")

monthly_df

,month,revenue,profit,profit_margin
0,2014-01-01 00:00:00+00:00,14236.90,2450.1907,17.21
1,2014-02-01 00:00:00+00:00,4519.92,862.3084,19.08
2,2014-03-01 00:00:00+00:00,55691.04,498.7299,0.90
3,2014-04-01 00:00:00+00:00,28295.35,3488.8352,12.33
4,2014-05-01 00:00:00+00:00,23648.28,2738.7096,11.58
5,2014-06-01 00:00:00+00:00,34595.14,4976.5244,14.39
6,2014-07-01 00:00:00+00:00,33946.37,-841.4826,-2.48
7,2014-08-01 00:00:00+00:00,27909.47,5318.1050,19.05
8,2014-09-01 00:00:00+00:00,81777.34,8328.0994,10.18
9,2014-10-01 00:00:00+00:00,31453.37,3448.2573,10.96


# Top 10 loss-making Customers

In [26]:
customer_loss_df = db_utils.run_pg_query("""
SELECT
    c.customer_name,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.customers c
JOIN superstore.orders o
    ON c.customer_id = o.customer_id
JOIN superstore.order_items i
    ON o.order_id = i.order_id
GROUP BY
    c.customer_id,
    c.customer_name
HAVING
    SUM(i.profit) < 0
ORDER BY
    profit ASC
LIMIT 10;
""")

customer_loss_df

,customer_name,revenue,profit,profit_margin
0,Cindy Stewart,5690.07,-6626.3895,-116.46
1,Grant Thornton,9351.20,-4108.6589,-43.94
2,Luke Foster,3930.52,-3583.9770,-91.18
3,Sharelle Roach,3233.48,-3333.9144,-103.11
4,Henry Goldwyn,3247.65,-2797.9635,-86.15
5,Nathan Cano,2218.99,-2204.8072,-99.36
6,Sean Braxton,8057.89,-2082.7451,-25.85
7,Sean Miller,25043.07,-1980.7393,-7.91
8,Christine Phan,5888.30,-1850.3029,-31.42
9,Natalie Fritzler,8322.82,-1695.9714,-20.38


# Loss by category

In [27]:
category_loss_df = db_utils.run_pg_query("""
SELECT
    p.category,
    SUM(i.profit) AS loss
FROM superstore.order_items i
JOIN superstore.products p
    ON i.product_key = p.product_key
GROUP BY
    p.category
HAVING
    SUM(i.profit) < 0
ORDER BY
    loss ASC;
""")

category_loss_df

,category,loss


## Top 10 customers by revenue

In [35]:
top_revenue_customers_df = db_utils.run_pg_query("""
SELECT
    c.customer_name,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.customers c
JOIN superstore.orders o
    ON c.customer_id = o.customer_id
JOIN superstore.order_items i
    ON o.order_id = i.order_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    revenue DESC
LIMIT 10;
""")

top_revenue_customers_df

,customer_name,revenue,profit,profit_margin
0,Sean Miller,25043.07,-1980.7393,-7.91
1,Tamara Chand,19052.22,8981.3239,47.14
2,Raymond Buch,15117.35,6976.0959,46.15
3,Tom Ashbrook,14595.62,4703.7883,32.23
4,Adrian Barton,14473.57,5444.8055,37.62
5,Ken Lonsdale,14175.23,806.8550,5.69
6,Sanjit Chand,14142.34,5757.4119,40.71
7,Hunter Lopez,12873.30,5622.4292,43.68
8,Sanjit Engle,12209.44,2650.6769,21.71
9,Christopher Conant,12129.08,2177.0493,17.95


## Top 10 customers by profit

In [36]:
top_profit_customers_df = db_utils.run_pg_query("""
SELECT
    c.customer_name,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.customers c
JOIN superstore.orders o
    ON c.customer_id = o.customer_id
JOIN superstore.order_items i
    ON o.order_id = i.order_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    profit DESC
LIMIT 10;
""")

top_profit_customers_df

,customer_name,revenue,profit,profit_margin
0,Tamara Chand,19052.22,8981.3239,47.14
1,Raymond Buch,15117.35,6976.0959,46.15
2,Sanjit Chand,14142.34,5757.4119,40.71
3,Hunter Lopez,12873.30,5622.4292,43.68
4,Adrian Barton,14473.57,5444.8055,37.62
5,Tom Ashbrook,14595.62,4703.7883,32.23
6,Christopher Martinez,8954.01,3899.8904,43.55
7,Keith Dawkins,8181.24,3038.6254,37.14
8,Andy Reiter,6608.45,2884.6208,43.65
9,Daniel Raglin,8350.87,2869.0760,34.36


## Regional performance

In [37]:
region_df = db_utils.run_pg_query("""
SELECT
    o.region,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.orders o
JOIN superstore.order_items i
    ON o.order_id = i.order_id
GROUP BY
    o.region
ORDER BY
    profit DESC;
""")

region_df

,region,revenue,profit,profit_margin
0,West,725457.93,108418.4489,14.94
1,East,678781.36,91522.7800,13.48
2,South,391721.90,46749.4303,11.93
3,Central,501239.88,39706.3625,7.92


## State Performance

In [38]:
state_df = db_utils.run_pg_query("""
SELECT
    o.state,
    SUM(i.sales) AS revenue,
    SUM(i.profit) AS profit,
    ROUND(
        SUM(i.profit) / NULLIF(SUM(i.sales), 0) * 100,
        2
    ) AS profit_margin
FROM superstore.orders o
JOIN superstore.order_items i
    ON o.order_id = i.order_id
GROUP BY
    o.state
ORDER BY
    profit DESC;
""")

state_df

,state,revenue,profit,profit_margin
0,California,457687.68,76381.3871,16.69
1,New York,310876.20,74038.5486,23.82
2,Washington,138641.29,33402.6517,24.09
3,Michigan,76269.61,24463.1876,32.07
4,Virginia,70636.72,18597.9504,26.33
5,Indiana,53555.36,18382.9363,34.33
6,Georgia,49095.84,16250.0433,33.10
7,Kentucky,36591.75,11199.6966,30.61
8,Minnesota,29863.15,10823.1874,36.24
9,Delaware,27451.07,9977.3748,36.35


In [28]:
print("KPI:", kpi_df.shape)
print("Category:", category_df.shape)
print("Furniture:", furniture_df.shape)
print("Monthly:", monthly_df.shape)
print("Customer Loss:", customer_loss_df.shape)
print("Category Loss:", category_loss_df.shape)

KPI: (1, 3)
Category: (3, 4)
Furniture: (4, 4)
Monthly: (48, 4)
Customer Loss: (10, 4)
Category Loss: (0, 2)


In [29]:
display(kpi_df)
display(category_df)
display(furniture_df)
display(customer_loss_df)

,total_revenue,total_profit,profit_margin
0,2297201.07,286397.0217,12.47


,category,revenue,profit,profit_margin
0,Technology,836154.10,145454.9481,17.40
1,Office Supplies,719046.99,122490.8008,17.04
2,Furniture,741999.98,18451.2728,2.49


,sub_category,revenue,profit,profit_margin
0,Tables,206965.68,-17725.4811,-8.56
1,Bookcases,114880.05,-3472.5560,-3.02
2,Furnishings,91705.12,13059.1436,14.24
3,Chairs,328449.13,26590.1663,8.10


,customer_name,revenue,profit,profit_margin
0,Cindy Stewart,5690.07,-6626.3895,-116.46
1,Grant Thornton,9351.20,-4108.6589,-43.94
2,Luke Foster,3930.52,-3583.9770,-91.18
3,Sharelle Roach,3233.48,-3333.9144,-103.11
4,Henry Goldwyn,3247.65,-2797.9635,-86.15
5,Nathan Cano,2218.99,-2204.8072,-99.36
6,Sean Braxton,8057.89,-2082.7451,-25.85
7,Sean Miller,25043.07,-1980.7393,-7.91
8,Christine Phan,5888.30,-1850.3029,-31.42
9,Natalie Fritzler,8322.82,-1695.9714,-20.38


# first executive visual: KPI cards.

In [30]:
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="number",
    value=kpi_df.loc[0, "total_revenue"],
    title={"text": "Total Revenue"},
    number={
        "prefix": "$",
        "valueformat": ",.2f"
    },
    domain={"row": 0, "column": 0}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=kpi_df.loc[0, "total_profit"],
    title={"text": "Total Profit"},
    number={
        "prefix": "$",
        "valueformat": ",.2f"
    },
    domain={"row": 0, "column": 1}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=kpi_df.loc[0, "profit_margin"],
    title={"text": "Profit Margin"},
    number={
        "suffix": "%",
        "valueformat": ".2f"
    },
    domain={"row": 0, "column": 2}
))

fig.update_layout(
    grid={
        "rows": 1,
        "columns": 3
    },
    height=250,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()

## Category profitability

In [31]:
fig = px.bar(
    category_df,
    x="category",
    y="profit",
    text="profit",
    title="Profit by Product Category"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    yaxis_title="Total Profit ($)",
    xaxis_title="Category"
)

fig.show()

# Furniture investigation

In [32]:
fig = px.bar(
    furniture_df,
    x="sub_category",
    y="profit",
    text="profit",
    title="Furniture Sub-Category Profitability"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.add_hline(
    y=0,
    line_width=1
)

fig.update_layout(
    yaxis_title="Total Profit ($)",
    xaxis_title="Furniture Sub-Category"
)

fig.show()

# Monthly Performance

In [33]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly_df["month"],
    y=monthly_df["revenue"],
    mode="lines+markers",
    name="Revenue"
))

fig.add_trace(go.Scatter(
    x=monthly_df["month"],
    y=monthly_df["profit"],
    mode="lines+markers",
    name="Profit"
))

fig.update_layout(
    title="Monthly Revenue and Profit Trend",
    xaxis_title="Month",
    yaxis_title="Amount ($)",
    hovermode="x unified"
)

fig.show()

# Top Loss-Making Customers

In [34]:
fig = px.bar(
    customer_loss_df.sort_values("profit"),
    x="profit",
    y="customer_name",
    orientation="h",
    text="profit",
    title="Top 10 Loss-Making Customers"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.add_vline(
    x=0,
    line_width=1
)

fig.update_layout(
    xaxis_title="Total Profit ($)",
    yaxis_title="Customer"
)

fig.show()

In [44]:
kpi_df

,Total Revenue,Total Profit,Profit Margin %,Total Orders
0,2297201.07,286397.0217,12.47,5009


In [45]:
total_revenue = kpi_df["Total Revenue"].iloc[0]
total_profit = kpi_df["Total Profit"].iloc[0]
profit_margin = kpi_df["Profit Margin %"].iloc[0]
total_orders = kpi_df["Total Orders"].iloc[0]

In [46]:
print(total_revenue)
print(total_profit)
print(profit_margin)
print(total_orders)

2297201.07
286397.0217
12.47
5009


In [47]:
import plotly.graph_objects as go

In [48]:
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="number",
    value=total_revenue,
    title={"text": "Total Revenue"},
    number={"prefix": "$", "valueformat": ",.2f"},
    domain={"row": 0, "column": 0}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=total_profit,
    title={"text": "Total Profit"},
    number={"prefix": "$", "valueformat": ",.2f"},
    domain={"row": 0, "column": 1}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=profit_margin,
    title={"text": "Profit Margin"},
    number={"suffix": "%", "valueformat": ".2f"},
    domain={"row": 0, "column": 2}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=total_orders,
    title={"text": "Total Orders"},
    number={"valueformat": ","},
    domain={"row": 0, "column": 3}
))

fig.update_layout(
    grid={"rows": 1, "columns": 4},
    height=180,
    margin={"l": 20, "r": 20, "t": 40, "b": 20}
)

fig.show()

In [49]:
category_df

,category,revenue,profit,profit_margin
0,Technology,836154.10,145454.9481,17.40
1,Office Supplies,719046.99,122490.8008,17.04
2,Furniture,741999.98,18451.2728,2.49


In [50]:
import plotly.express as px

fig = px.bar(
    category_df,
    x="category",
    y="profit",
    title="Profit by Category",
    text="profit"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Total Profit",
    template="plotly_white",
    height=450
)

fig.show()

In [51]:
fig = px.bar(
    furniture_df,
    x="sub_category",
    y="profit",
    title="Furniture Profit by Sub-Category",
    text="profit"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=450
)

fig.show()

In [52]:
fig = px.line(
    monthly_df,
    x="month",
    y=["revenue", "profit"],
    title="Monthly Revenue vs Profit",
    markers=True
)

fig.update_layout(
    template="plotly_white",
    height=450,
    xaxis_title="Month",
    yaxis_title="Amount ($)"
)

fig.show()

In [56]:
fig = px.bar(
    orders_category_df,
    x="Category",
    y="Total Orders",
    title="Orders by Category",
    text="Total Orders"
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=400
)

fig.show()

In [62]:
fig = px.bar(
    region_df,
    x="region",
    y=["revenue", "profit"],
    barmode="group",
    title="Revenue vs Profit by Region"
)

fig.update_layout(
    template="plotly_white",
    height=400,
    yaxis_title="Amount ($)",
    xaxis_title="Region"
)

fig.show()

In [68]:
fig = px.bar(
    state_df.sort_values("profit"),
    x="profit",
    y="state",
    orientation="h",
    title="Profit Performance by State",
    text="profit"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=700,
    xaxis_title="Total Profit",
    yaxis_title="State"
)

fig.show()

In [70]:
fig = px.bar(
    top_revenue_customers_df.sort_values("revenue"),
    x="revenue",
    y="customer_name",
    orientation="h",
    title="Top 10 Customers by Revenue",
    text="revenue"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=500,
    xaxis_title="Revenue ($)",
    yaxis_title="Customer"
)

fig.show()

In [72]:
fig = px.bar(
    top_profit_customers_df.sort_values("profit"),
    x="profit",
    y="customer_name",
    orientation="h",
    title="Top 10 Customers by Profit",
    text="profit"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=500,
    xaxis_title="Profit ($)",
    yaxis_title="Customer"
)

fig.show()

In [74]:
fig = px.bar(
    customer_loss_df.sort_values("profit"),
    x="profit",
    y="customer_name",
    orientation="h",
    title="Top 10 Loss-Making Customers",
    text="profit"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=500,
    xaxis_title="Profit ($)",
    yaxis_title="Customer"
)

fig.show()

In [76]:
fig = px.bar(
    orders_region_df.sort_values("total_orders"),
    x="total_orders",
    y="region",
    orientation="h",
    title="Orders by Region",
    text="total_orders"
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=400,
    xaxis_title="Total Orders",
    yaxis_title="Region"
)

fig.show()

In [78]:
fig = px.bar(
    orders_state_df.head(15).sort_values("total_orders"),
    x="total_orders",
    y="state",
    orientation="h",
    title="Top 15 States by Orders",
    text="total_orders"
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=550,
    xaxis_title="Total Orders",
    yaxis_title="State"
)

fig.show()

In [80]:
fig = px.bar(
    orders_customer_df.sort_values("total_orders"),
    x="total_orders",
    y="customer_name",
    orientation="h",
    title="Top 10 Customers by Order Volume",
    text="total_orders"
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=500,
    xaxis_title="Total Orders",
    yaxis_title="Customer"
)

fig.show()

In [81]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [82]:
total_revenue = kpi_df.loc[0, "Total Revenue"]
total_profit = kpi_df.loc[0, "Total Profit"]
profit_margin = kpi_df.loc[0, "Profit Margin %"]
total_orders = kpi_df.loc[0, "Total Orders"]

In [83]:
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="number",
    value=total_revenue,
    number={"prefix": "$", "valueformat": ",.0f"},
    title={"text": "Total Revenue"},
    domain={"x": [0, 0.25], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=total_profit,
    number={"prefix": "$", "valueformat": ",.0f"},
    title={"text": "Total Profit"},
    domain={"x": [0.25, 0.5], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=profit_margin,
    number={"suffix": "%", "valueformat": ".2f"},
    title={"text": "Profit Margin"},
    domain={"x": [0.5, 0.75], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=total_orders,
    number={"valueformat": ",.0f"},
    title={"text": "Total Orders"},
    domain={"x": [0.75, 1], "y": [0, 1]}
))

fig.update_layout(
    template="plotly_white",
    height=180,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

In [84]:
fig = px.bar(
    category_df.sort_values("profit"),
    x="profit",
    y="category",
    orientation="h",
    title="Profit by Category",
    text="profit"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=400,
    xaxis_title="Profit ($)",
    yaxis_title="Category"
)

fig.show()

In [4]:
# ============================================================
# EXECUTIVE DASHBOARD
# ============================================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px


# ============================================================
# 1. CREATE DASHBOARD
# ============================================================

fig = make_subplots(
    rows=7,
    cols=2,

    # KPI row
    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"}
        ],
        [
            {"type": "indicator"},
            {"type": "indicator"}
        ],

        # Charts
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy", "colspan": 2}, None],
    ],

    vertical_spacing=0.07,

    subplot_titles=(
        "",
        "",
        "",
        "",

        "Monthly Revenue vs Profit",
        "Profit by Category",

        "Orders by Category",
        "Furniture Profit by Sub-Category",

        "Top 10 Customers by Revenue",
        "Top 10 Customers by Profit",

        "Top 10 Loss-Making Customers",
        "",

        "State Profitability",
        ""
    )
)


# ============================================================
# 2. KPI CARDS
# ============================================================

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpi.loc[0, "Total Revenue"],
        title={"text": "TOTAL REVENUE"},
        number={
            "prefix": "$",
            "valueformat": ",.2f"
        }
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpi.loc[0, "Total Profit"],
        title={"text": "TOTAL PROFIT"},
        number={
            "prefix": "$",
            "valueformat": ",.2f"
        }
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpi.loc[0, "Profit Margin %"],
        title={"text": "PROFIT MARGIN"},
        number={
            "suffix": "%",
            "valueformat": ".2f"
        }
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpi.loc[0, "Total Orders"],
        title={"text": "TOTAL ORDERS"},
        number={
            "valueformat": ","
        }
    ),
    row=2,
    col=2
)


# ============================================================
# 3. MONTHLY REVENUE VS PROFIT
# ============================================================

fig.add_trace(
    go.Scatter(
        x=monthly_df["Month"],
        y=monthly_df["Revenue"],
        mode="lines+markers",
        name="Revenue"
    ),
    row=3,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=monthly_df["Month"],
        y=monthly_df["Profit"],
        mode="lines+markers",
        name="Profit"
    ),
    row=3,
    col=1
)


# ============================================================
# 4. PROFIT BY CATEGORY
# ============================================================

fig.add_trace(
    go.Bar(
        x=category_df["category"],
        y=category_df["profit"],
        text=category_df["profit"],
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        name="Profit"
    ),
    row=3,
    col=2
)


# ============================================================
# 5. ORDERS BY CATEGORY
# ============================================================

fig.add_trace(
    go.Bar(
        x=orders_category_df["Category"],
        y=orders_category_df["Total Orders"],
        text=orders_category_df["Total Orders"],
        textposition="outside",
        name="Orders"
    ),
    row=4,
    col=1
)


# ============================================================
# 6. FURNITURE
# ============================================================

fig.add_trace(
    go.Bar(
        x=furniture_df["Sub-Category"],
        y=furniture_df["Total Profit"],
        text=furniture_df["Total Profit"],
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        name="Furniture Profit"
    ),
    row=4,
    col=2
)


# ============================================================
# 7. TOP REVENUE CUSTOMERS
# ============================================================

fig.add_trace(
    go.Bar(
        x=top_revenue_customers_df["revenue"],
        y=top_revenue_customers_df["customer_name"],
        orientation="h",
        text=top_revenue_customers_df["revenue"],
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        name="Revenue"
    ),
    row=5,
    col=1
)


# ============================================================
# 8. TOP PROFIT CUSTOMERS
# ============================================================

fig.add_trace(
    go.Bar(
        x=top_profit_customers_df["profit"],
        y=top_profit_customers_df["customer_name"],
        orientation="h",
        text=top_profit_customers_df["profit"],
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        name="Profit"
    ),
    row=5,
    col=2
)


# ============================================================
# 9. LOSS-MAKING CUSTOMERS
# ============================================================

fig.add_trace(
    go.Bar(
        x=customer_loss_df["Total Profit"],
        y=customer_loss_df["Customer Name"],
        orientation="h",
        text=customer_loss_df["Total Profit"],
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        name="Loss"
    ),
    row=6,
    col=1
)


# ============================================================
# 10. STATE PERFORMANCE
# ============================================================

state_sorted = state_df.sort_values(
    "profit",
    ascending=True
)

fig.add_trace(
    go.Bar(
        x=state_sorted["profit"],
        y=state_sorted["state"],
        orientation="h",
        text=state_sorted["profit"],
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        name="State Profit"
    ),
    row=7,
    col=1
)


# ============================================================
# 11. TITLES / AXES
# ============================================================

fig.update_xaxes(
    title_text="Month",
    row=3,
    col=1
)

fig.update_yaxes(
    title_text="Amount ($)",
    row=3,
    col=1
)

fig.update_xaxes(
    title_text="Profit ($)",
    row=3,
    col=2
)

fig.update_yaxes(
    title_text="Profit ($)",
    row=4,
    col=2
)

fig.update_xaxes(
    title_text="Revenue ($)",
    row=5,
    col=1
)

fig.update_xaxes(
    title_text="Profit ($)",
    row=5,
    col=2
)

fig.update_xaxes(
    title_text="Profit ($)",
    row=6,
    col=1
)

fig.update_xaxes(
    title_text="Profit ($)",
    row=7,
    col=1
)


# ============================================================
# 12. SORT HORIZONTAL BAR CHARTS
# ============================================================

fig.update_yaxes(
    categoryorder="total ascending",
    row=5,
    col=1
)

fig.update_yaxes(
    categoryorder="total ascending",
    row=5,
    col=2
)

fig.update_yaxes(
    categoryorder="total descending",
    row=6,
    col=1
)


# ============================================================
# 13. DASHBOARD TITLE
# ============================================================

fig.update_layout(
    title={
        "text": "SUPERSTORE EXECUTIVE BUSINESS PERFORMANCE DASHBOARD",
        "x": 0.5,
        "xanchor": "center",
        "font": {
            "size": 24
        }
    },

    template="plotly_white",

    height=2800,

    showlegend=False,

    margin=dict(
        l=80,
        r=80,
        t=100,
        b=80
    )
)


# ============================================================
# 14. DISPLAY
# ============================================================

fig.show()

NameError: name 'kpi' is not defined